In [1]:
from dataclasses import dataclass
from datetime import datetime, timedelta


@dataclass
class ConfidenceInput:
    total_fields: int
    filled_fields: int
    query_keywords: list[str]
    data_last_updated: datetime


def calculate_confidence(input: ConfidenceInput) -> dict:
    """
    Returns confidence score (0-100) with breakdown.
    Weights: completeness=40%, clarity=35%, freshness=25%
    """

    # 1. Data Completeness (0-100)
    if input.total_fields == 0:
        completeness = 0.0
    else:
        completeness = (input.filled_fields / input.total_fields) * 100

    # 2. Query Clarity (0-100)
    #    Simple heuristic: more keywords = clearer intent
    #    TODO: replace with NLP-based intent classifier in Week 5-6
    keyword_count = len(input.query_keywords)
    if keyword_count >= 3:
        clarity = 100.0
    elif keyword_count == 2:
        clarity = 70.0
    elif keyword_count == 1:
        clarity = 40.0
    else:
        clarity = 0.0

    # 3. Data Freshness (0-100)
    age_days = (datetime.now() - input.data_last_updated).days
    if age_days <= 7:
        freshness = 100.0
    elif age_days <= 30:
        freshness = 75.0
    elif age_days <= 90:
        freshness = 50.0
    elif age_days <= 365:
        freshness = 25.0
    else:
        freshness = 10.0

    # Weighted final score
    W_COMPLETENESS = 0.40
    W_CLARITY = 0.35
    W_FRESHNESS = 0.25

    final_score = (
        completeness * W_COMPLETENESS
        + clarity * W_CLARITY
        + freshness * W_FRESHNESS
    )

    return {
        "score": round(final_score, 1),
        "breakdown": {
            "completeness": round(completeness, 1),
            "clarity": round(clarity, 1),
            "freshness": round(freshness, 1),
        },
        "label": _score_to_label(final_score),
    }


def _score_to_label(score: float) -> str:
    if score >= 80:
        return "high"
    elif score >= 50:
        return "medium"
    else:
        return "low"


# --- Example Usage ---
if __name__ == "__main__":
    result = calculate_confidence(
        ConfidenceInput(
            total_fields=10,
            filled_fields=9,
            query_keywords=["harga", "paket", "A"],
            data_last_updated=datetime.now() - timedelta(days=3),
        )
    )
    print(result)
    # {'score': 91.0, 'breakdown': {'completeness': 90.0, 'clarity': 100.0, 'freshness': 100.0}, 'label': 'high'}

    result_low = calculate_confidence(
        ConfidenceInput(
            total_fields=10,
            filled_fields=3,
            query_keywords=["bagus"],
            data_last_updated=datetime.now() - timedelta(days=400),
        )
    )
    print(result_low)
    # {'score': 28.6, 'breakdown': {'completeness': 30.0, 'clarity': 40.0, 'freshness': 10.0}, 'label': 'low'}

{'score': 96.0, 'breakdown': {'completeness': 90.0, 'clarity': 100.0, 'freshness': 100.0}, 'label': 'high'}
{'score': 28.5, 'breakdown': {'completeness': 30.0, 'clarity': 40.0, 'freshness': 10.0}, 'label': 'low'}
